<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Intel System Console Hardware-in-the-Loop

This tutorial transfers numpy data through an Intel JTAG-to-Avalon master and exchanges text commands through JTAG UART. It requires Quartus System Console, a programmed FPGA, and matching services in the design.

## 1. Persistent session

`IntelSystemConsoleSession` starts one persistent `system-console` process and communicates with the packaged Tcl script. Reusing the process avoids Quartus startup overhead for every transfer. `JtagSession` is a backward-compatible alias of the same class.

`master_index` and `uart_index` select services from the lists discovered by System Console. `startup_timeout` limits initial connection time, and `work_dir` selects where temporary binary transfer files are placed.

In [ ]:
import numpy as np

from fpga_verification.hil.intel import IntelSystemConsoleSession, JtagSession

frame = np.arange(64, dtype=np.uint16).reshape(8, 8)

with IntelSystemConsoleSession(
    system_console="system-console",
    master_index=0,
    uart_index=0,
    startup_timeout=30.0,
    work_dir=".",
) as hw:
    hw.write_memory(frame, address=0x01000000)
    response = hw.command("status", timeout=3.0)
    frame_out = hw.read_memory(frame.shape, address=0x01000000)

print(response)
print(frame_out.shape)

## 2. Memory transfers

`write_memory(data, address, chunk_size=4096)` converts the input to little-endian unsigned 16-bit words and transfers the resulting file in chunks. `read_memory(shape, address, chunk_size=4096)` reads a two-dimensional array of little-endian `uint16` values and validates the returned word count. Progress is displayed with `tqdm`.

The current API is intentionally specialized: memory elements are always 16-bit, and `read_memory()` expects a two-dimensional shape. Use `VideoFormat` and `FrameSize` before HIL transfer when the data represents a video frame, and verify that the hardware memory layout uses the same component order.

In [ ]:
def memory_round_trip(hw, source, address):
    hw.write_memory(source, address=address, chunk_size=16 * 1024)
    result = hw.read_memory(
        source.shape,
        address=address,
        chunk_size=16 * 1024,
    )
    np.testing.assert_array_equal(result, source.astype("<u2"))
    return result

## 3. JTAG UART commands

`command(text, timeout=3.0, debug=False)` sends UTF-8 text through the selected JTAG UART and returns the first non-empty response line. `debug=True` prints raw System Console protocol lines. The FPGA software must implement the corresponding command protocol; the library only transports text.

In [ ]:
def query_firmware(hw):
    version = hw.command("version", timeout=3.0)
    status = hw.command("status", timeout=3.0, debug=True)
    return version, status

## 4. Opening and closing

The context manager calls `open()` and guarantees `close()` on normal exit or exceptions. Manual lifecycle is also supported:

```python
hw = IntelSystemConsoleSession(...)
try:
    hw.open()
    ...
finally:
    hw.close()
```

Calls are protected by a reentrant lock, so one process cannot interleave commands from multiple Python threads. `close()` first requests a clean Tcl shutdown, then terminates or kills an unresponsive process.

## 5. Operational cautions

The session creates `data_in.bin` and `data_out.bin` in `work_dir`, replacing files with those names. Use a dedicated writable directory for concurrent tests. Confirm the selected service indices and target address before writing: this is real hardware access and an incorrect address may modify control registers or active buffers. Startup, UART timeout, process termination, malformed responses, and short memory reads are reported as Python exceptions.